# Phase 3: identity training, lockbox test, H1/H2

Run on Google Colab with Google Drive mounted. Each phase starts only if the previous phase's gate file says GO. See the repository README for the full workflow.

### Cell 3.0: Start the session

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, shutil, importlib, time, math, hashlib, datetime
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT = Path("/content/drive/MyDrive/ecg_leak")
sys.path.insert(0, str(PROJECT / "src"))
import ecgleak.config as ecfg, ecgleak.models as M, ecgleak.train as TR, ecgleak.evaluate as EV
for mod in (ecfg, M, TR, EV):
    importlib.reload(mod)
CFG = ecfg.load(str(PROJECT / "config.yaml"))
PATHS = CFG["paths"]
gate2 = json.load(open(f"{PATHS['gates']}/gate_2.json"))
assert gate2["go"], "Phase 2 gate is not GO: finish Phase 2 first"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW = int(CFG["preprocess"]["window_s"] * CFG["data"]["fs"])

LOCAL_CACHE = "/content/cache"
os.makedirs(LOCAL_CACHE, exist_ok=True)
splits = pd.read_csv(f"{PATHS['splits']}/splits.csv", index_col="ecg_id")
def load_split(s):
    dst = f"{LOCAL_CACHE}/{s}.npz"
    if not os.path.isfile(dst):
        shutil.copyfile(f"{PATHS['cache']}/{s}.npz", dst)
    z = np.load(dst, allow_pickle=False)
    d = {k: z[k] for k in z.files}
    d["date"] = pd.to_datetime(splits.loc[d["ecg_id"], "recording_date"]).to_numpy()
    return d
D = {s: load_split(s) for s in ["id_train", "id_eval", "probe_dev"]}     # lockbox: only in cell 3.6
SUPER = list(D["probe_dev"]["label_names"])
print("Phase 2 gate: GO | Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU")
for s, d in D.items():
    print(f"  {s:10s} {len(d['ecg_id']):6,} records")

### Cell 3.1: Lock the rules before any result

In [ ]:
PLAN = {
    "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "primary_task": f"{CFG['leakage']['primary']} vs NORM",
    "identity_check": "best identity EER on ID-eval <= IDENTITY_MAX_RATIO x median-beat baseline EER",
    "IDENTITY_MAX_RATIO": 0.80,               # agree this with your supervisor BEFORE running 3.2
    "ci_halfwidth_target": CFG["stats"]["ci_halfwidth_target"],
    "H1": "Delta_rand > 0 and its 95% CI excludes 0",
    "H2": "Delta_raw > 0 and Delta_demo > 0, both CIs exclude 0",
    "probe": "standardise -> L2 logistic regression; C from C_grid by 5-fold CV on probe-dev folds; refit on all probe-dev",
    "raw_beat": "median beat -> PCA 64 (fit on probe-dev)", "template": "mean of 4 non-overlapping 2.5 s windows, L2-normalised",
    "C_grid": CFG["leakage"]["C_grid"], "untrained_seeds": CFG["leakage"]["untrained_seeds"],
    "n_boot": CFG["stats"]["n_boot"], "n_perm": CFG["stats"]["n_perm"], "seed": CFG["seed"],
}
plan_path = f"{PATHS['gates']}/phase3_plan.json"
if os.path.isfile(plan_path):
    old = json.load(open(plan_path))
    changed = {k for k in PLAN if k != "written_at" and old.get(k) != PLAN[k]}
    assert not changed, f"Plan already written on {old['written_at']}; these items differ: {changed}. Do not change rules after training."
    PLAN = old
    print("Plan already locked on", PLAN["written_at"])
else:
    json.dump(PLAN, open(plan_path, "w"), indent=2)
    print("Plan locked:", plan_path)
print(json.dumps({k: PLAN[k] for k in ["primary_task", "identity_check", "IDENTITY_MAX_RATIO", "H1", "H2"]}, indent=2))

### Cell 3.2: Train the full identity model

In [ ]:
MAIN_DIR = f"{PATHS['runs']}/id_main_s0"
idv = D["id_eval"]
def id_eval_fn(model):
    r = EV.verification(EV.embed(model, idv["signal"], WINDOW, DEVICE), idv["patient_id"], idv["date"])
    return {"eer": r["eer"], "rank1": r["rank1"]}

ck = TR.latest_checkpoint(MAIN_DIR)
last = torch.load(ck, map_location="cpu", weights_only=False) if ck else None
if last and (last["bad"] >= CFG["train"]["patience"] or last["epoch"] + 1 >= CFG["train"]["epochs"]):
    hist = last["history"]
    print(f"Training already finished earlier ({len(hist)} epochs): skipped.")
else:
    ecfg.freeze(CFG, MAIN_DIR)
    t0 = time.time()
    model, hist = TR.train_identity(CFG, D["id_train"]["signal"], D["id_train"]["patient_id"], MAIN_DIR, DEVICE,
                                    eval_fn=id_eval_fn)
    print(f"Training finished: {len(hist)} epochs in total, {(time.time() - t0) / 60:.1f} min this session")
pd.DataFrame(hist).to_csv(f"{MAIN_DIR}/history.csv", index=False)

### Cell 3.3: Identity check and training curves

In [ ]:
import matplotlib.pyplot as plt
id_model, st = TR.load_best(CFG, MAIN_DIR, DEVICE)
final_id = EV.verification(EV.embed(id_model, idv["signal"], WINDOW, DEVICE), idv["patient_id"], idv["date"])
base_eer = gate2["identity_baseline"]["eer"]
IDENTITY_OK = final_id["eer"] <= PLAN["IDENTITY_MAX_RATIO"] * base_eer
print(f"Identity model (best epoch {st['epoch']}): EER {final_id['eer']:.3f} | rank-1 {final_id['rank1']:.3f}")
print(f"Median-beat baseline: EER {base_eer:.3f} | rank-1 {gate2['identity_baseline']['rank1']:.3f}")
print(f"Identity check (EER <= {PLAN['IDENTITY_MAX_RATIO']} x baseline = {PLAN['IDENTITY_MAX_RATIO'] * base_eer:.3f}):",
      "PASS" if IDENTITY_OK else "FAIL: fix the identity model before measuring any leak")

h = pd.read_csv(f"{MAIN_DIR}/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(h["epoch"], h["loss"]); ax[0].set_title("Training loss (ArcFace)"); ax[0].set_xlabel("epoch")
ax[1].plot(h["epoch"], h["eer"], label="identity model"); ax[1].axhline(base_eer, ls="--", c="grey", label="median-beat baseline")
ax[1].set_title("Identity EER on unseen patients (lower is better)"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{PATHS['figures']}/phase3_identity_training.png", dpi=150); plt.show()

### Cell 3.4: Leak-measurement code: leakage.py

In [ ]:
%%writefile /content/drive/MyDrive/ecg_leak/src/ecgleak/leakage.py
import warnings
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

def fit_and_score(X_dev, y_dev, folds, X_test, C_grid, pca=None, seed=0):
    """Choose C by CV over the probe-dev patient folds, refit on all probe-dev, score the test set once."""
    cv = [(np.where(folds != f)[0], np.where(folds == f)[0]) for f in np.unique(folds)]
    steps = [StandardScaler()]
    if pca:
        steps.append(PCA(n_components=min(pca, X_dev.shape[1]), random_state=seed))
    steps.append(LogisticRegressionCV(Cs=list(C_grid), cv=cv, scoring="roc_auc", max_iter=5000))
    model = make_pipeline(*steps).fit(X_dev, y_dev)
    return model.predict_proba(X_test)[:, 1], float(model[-1].C_[0])

def auc(y, s):
    return float(roc_auc_score(y, s))

def paired_bootstrap(y, scores, n_boot=2000, seed=0):
    """scores: dict name -> test-set scores. Resamples patients (one record each) and recomputes every AUROC."""
    rng = np.random.default_rng(seed)
    n, out = len(y), {k: [] for k in scores}
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() == y[i].max():
            continue
        for k, s in scores.items():
            out[k].append(roc_auc_score(y[i], s[i]))
    return {k: np.array(v) for k, v in out.items()}

def percentile_rank(s):
    return (np.argsort(np.argsort(s)) + 0.5) / len(s)

def paired_permutation(y, s_a, s_b, n_perm=1000, seed=0):
    """Two-sided p-value for AUROC(a) - AUROC(b): swap the two scores within each patient at random."""
    rng = np.random.default_rng(seed)
    a, b = percentile_rank(s_a), percentile_rank(s_b)
    obs = roc_auc_score(y, a) - roc_auc_score(y, b)
    null = np.empty(n_perm)
    for j in range(n_perm):
        swap = rng.random(len(y)) < 0.5
        null[j] = roc_auc_score(y, np.where(swap, b, a)) - roc_auc_score(y, np.where(swap, a, b))
    return float((1 + np.sum(np.abs(null) >= abs(obs))) / (n_perm + 1))

### Cell 3.5: Freeze the model before opening the lockbox

In [ ]:
import ecgleak.leakage as LK
importlib.reload(LK)
pdv = D["probe_dev"]
keep_dev, y_dev = EV.task_data(pdv["labels"], SUPER, CFG["leakage"]["primary"])
folds_dev = pdv["fold"][keep_dev]

def file_sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()

ready = {"plan": PLAN, "identity_ok": IDENTITY_OK, "identity_eer": final_id["eer"], "baseline_eer": base_eer,
         "model_file": f"{MAIN_DIR}/best.pt", "model_sha256": file_sha(f"{MAIN_DIR}/best.pt"),
         "best_epoch": st["epoch"], "splits_sha256": file_sha(f"{PATHS['splits']}/splits.csv"),
         "n_dev": int(keep_dev.sum()), "frozen_at": datetime.datetime.now().isoformat(timespec="seconds")}
assert IDENTITY_OK, "Identity check failed (cell 3.3): do NOT open the lockbox. Improve the identity model first."
ready_path = f"{PATHS['gates']}/gate_3_ready.json"
if os.path.isfile(ready_path):
    prev = json.load(open(ready_path))
    assert prev["model_sha256"] == ready["model_sha256"], \
        "A different model was frozen before. The lockbox result must come from the first frozen model."
    ready = prev
    print("Already frozen on", ready["frozen_at"], "(same model): continuing")
else:
    json.dump(ready, open(ready_path, "w"), indent=2, default=str)
    print("Frozen:", ready_path)
print(f"Model sha256 {ready['model_sha256'][:16]}... | best epoch {ready['best_epoch']} | probe-dev records for the task {ready['n_dev']:,}")

### Cell 3.6: Open the lockbox and score every representation

In [ ]:
LOCK = load_split("lockbox")
keep_lb, y_lb = EV.task_data(LOCK["labels"], SUPER, CFG["leakage"]["primary"])
print(f"Lockbox opened: {PLAN['primary_task']} | positives {y_lb.sum():,} | NORM-only {(1 - y_lb).sum():,}")

def reps(d, keep):
    sig = d["signal"][keep]
    out = {"identity_template": EV.embed(id_model, sig, WINDOW, DEVICE)}
    for k, m in enumerate(M.untrained_copies(CFG, PLAN["untrained_seeds"], DEVICE)):
        out[f"untrained_{k}"] = EV.embed(m, sig, WINDOW, DEVICE)
    out["raw_median_beat"] = d["median_beat"][keep]
    out["age_sex"] = EV.demo_features(d["age"][keep], d["sex"][keep])
    out["template_plus_age_sex"] = np.c_[out["identity_template"], out["age_sex"]]
    return out

R_dev, R_lb = reps(pdv, keep_dev), reps(LOCK, keep_lb)
scores, chosen_C = {}, {}
for name in R_dev:
    scores[name], chosen_C[name] = LK.fit_and_score(R_dev[name], y_dev, folds_dev, R_lb[name], PLAN["C_grid"],
                                                    pca=64 if name == "raw_median_beat" else None)
aucs = {k: LK.auc(y_lb, s) for k, s in scores.items()}
np.savez(f"{PATHS['results']}/lockbox_scores_{CFG['leakage']['primary']}.npz", y=y_lb,
         ecg_id=LOCK["ecg_id"][keep_lb], **scores)
print(pd.Series(aucs, name="lockbox AUROC").round(4).to_string())

### Cell 3.7: Three gains with CIs and p-values

In [ ]:
untr = [k for k in scores if k.startswith("untrained_")]
boot = LK.paired_bootstrap(y_lb, scores, PLAN["n_boot"], seed=CFG["seed"])
def ci(x):
    return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

d_rand_boot = boot["identity_template"] - np.mean([boot[k] for k in untr], axis=0)
deltas = {
    "Delta_rand": (aucs["identity_template"] - np.mean([aucs[k] for k in untr]), d_rand_boot),
    "Delta_raw": (aucs["identity_template"] - aucs["raw_median_beat"], boot["identity_template"] - boot["raw_median_beat"]),
    "Delta_demo": (aucs["template_plus_age_sex"] - aucs["age_sex"], boot["template_plus_age_sex"] - boot["age_sex"]),
}
pooled_untrained = np.mean([LK.percentile_rank(scores[k]) for k in untr], axis=0)
pvals = {"Delta_rand": LK.paired_permutation(y_lb, scores["identity_template"], pooled_untrained, PLAN["n_perm"], CFG["seed"]),
         "Delta_raw": LK.paired_permutation(y_lb, scores["identity_template"], scores["raw_median_beat"], PLAN["n_perm"], CFG["seed"]),
         "Delta_demo": LK.paired_permutation(y_lb, scores["template_plus_age_sex"], scores["age_sex"], PLAN["n_perm"], CFG["seed"])}
table = pd.DataFrame([{"gain": k, "estimate": v[0], "ci_low": ci(v[1])[0], "ci_high": ci(v[1])[1],
                       "ci_halfwidth": (ci(v[1])[1] - ci(v[1])[0]) / 2, "p_perm": pvals[k]}
                      for k, v in deltas.items()]).set_index("gain")
table.to_csv(f"{PATHS['results']}/main_leakage_{CFG['leakage']['primary']}.csv")
print(f"{PLAN['primary_task']} on the lockbox ({len(y_lb):,} patients), {len(d_rand_boot)} bootstrap resamples:\n")
print(table.round(4).to_string())

### Cell 3.8: Figure 1 (main result)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 2.6))
yy = np.arange(len(table))[::-1]
ax.errorbar(table["estimate"], yy, xerr=[table["estimate"] - table["ci_low"], table["ci_high"] - table["estimate"]],
            fmt="o", capsize=4, color="black")
ax.axvline(0, color="grey", ls="--")
ax.set_yticks(yy); ax.set_yticklabels(table.index)
ax.set_xlabel("AUROC gain of the identity template (95% bootstrap CI)")
ax.set_title(f"Diagnostic leakage, {PLAN['primary_task']}, lockbox")
plt.tight_layout(); plt.savefig(f"{PATHS['figures']}/figure1_main_leakage_{CFG['leakage']['primary']}.png", dpi=200); plt.show()

### Cell 3.9: Phase 3 gate: H1 and H2

In [ ]:
row = table.to_dict("index")
excl0 = lambda k: row[k]["ci_low"] > 0
H1 = row["Delta_rand"]["estimate"] > 0 and excl0("Delta_rand")
H2 = row["Delta_raw"]["estimate"] > 0 and excl0("Delta_raw") and row["Delta_demo"]["estimate"] > 0 and excl0("Delta_demo")
PRECISE = row["Delta_rand"]["ci_halfwidth"] <= PLAN["ci_halfwidth_target"]
gate = {"Identity model beats baseline": IDENTITY_OK, "Settings frozen before lockbox": os.path.isfile(ready_path),
        "Lockbox scored once with frozen model": True,
        f"Delta_rand CI half-width <= {PLAN['ci_halfwidth_target']}": PRECISE}
print("PHASE 3 GATE\n" + "-" * 55)
for k, v in gate.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("-" * 55)
print(f"H1 (identity template leaks CD beyond untrained copies): {'SUPPORTED' if H1 else 'NOT SUPPORTED'}")
print(f"H2 (beyond raw heartbeat AND beyond age+sex):            {'SUPPORTED' if H2 else 'NOT SUPPORTED'}")
if not PRECISE:
    print("CI wider than the target: report the result as imprecise; do not add lockbox data or re-run with new settings.")
GO = gate["Identity model beats baseline"] and gate["Settings frozen before lockbox"]
json.dump({"phase": 3, "go": GO, "checks": gate, "H1_supported": H1, "H2_supported": H2, "precise": PRECISE,
           "identity": final_id, "aucs": aucs, "chosen_C": chosen_C, "gains": table.reset_index().to_dict("records"),
           "n_lockbox": int(len(y_lb)), "environment": ecfg.environment()},
          open(f"{PATHS['gates']}/gate_3.json", "w"), indent=2, default=str)
print("Saved gates/gate_3.json, results/main_leakage_*.csv and Figure 1.")